In [1]:
# %pip install tiktoken
# # # ssh -p 50413 root@115.231.176.132 -L 8080:localhost:8080
# # # ssh -p 25259 root@ssh8.vast.ai -L 8080:localhost:8080

In [2]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from typing import Any, Dict, Iterable, Optional
import math
import os
import regex as re
import tiktoken
import warnings
import random
warnings.filterwarnings('ignore')

print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: cuda


In [3]:
# Get the GPT-2 encoding --> GPT-2 does not merge spaces, so it's bad for coding
enc_gpt2 = tiktoken.get_encoding("gpt2")
enc_gpt4 = tiktoken.get_encoding("cl100k_base")

list_of_tokens = enc_gpt2.encode("hello world! how are you? I'm fine, thank you!😊")
print("Encoded tokens:", list_of_tokens)

decode_tokens = enc_gpt2.decode(list_of_tokens)
print("Decoded text:", decode_tokens)

list_of_tokens = enc_gpt4.encode("hello world! how are you? I'm fine, thank you!😊")
print("Encoded tokens:", list_of_tokens)

# gpt-4 merges spaces, so it's better for coding
list_of_tokens = enc_gpt4.encode("how are you?")
print("Encoded tokens from example:", list_of_tokens)
print(len(list_of_tokens))


Encoded tokens: [31373, 995, 0, 703, 389, 345, 30, 314, 1101, 3734, 11, 5875, 345, 0, 47249, 232]
Decoded text: hello world! how are you? I'm fine, thank you!😊
Encoded tokens: [15339, 1917, 0, 1268, 527, 499, 30, 358, 2846, 7060, 11, 9901, 499, 0, 76460, 232]
Encoded tokens from example: [5269, 527, 499, 30]
4


In [4]:
vocab_size = 50304 
print("Vocabulary size:", vocab_size)

Vocabulary size: 50304


In [5]:
class Linear(nn.Module):

    def __init__(self, features_in: int, features_out: int, bias = False):

        super().__init__()
        # Initialize the weight parameter with a linear weights where mean = 0, std = sqrt(2 / (features_in + features_out)) 
        self.weight = nn.Parameter(torch.empty(features_in, features_out))

        std = math.sqrt(2 / (features_in + features_out))
        torch.nn.init.normal_(self.weight, mean=0.0, std=std)

    def forward(self, x):
        self.out = x @ self.weight.transpose(-2, -1)
        return self.out

    def parameters(self):
        return [self.weight]



In [6]:
class Embedding(nn.Module):

    def __init__(self, num_embedding, embedding_dim):
        super().__init__()
        # Initialize the weight parameter with a linear weights where mean = 0, std = 1 
        self.weights = nn.Parameter(torch.randn(num_embedding, embedding_dim).to(device))

    def forward(self, idx):
        self.out = self.weights[idx]
        return self.out

    def parameters(self):
        return [self.weights]


In [7]:
class RMSNorm(nn.Module):

    def __init__(self, d_model: int, eps: float = 1e-5):

        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model).to(device))
        self.eps = eps

    def forward(self, x):

        in_dtype = x.dtype
        x = x.to(torch.float32)
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        normalized_rmsx = x / rms
        result = normalized_rmsx * self.gamma
        return result.to(in_dtype)


In [8]:
class SwiGLU_FeedForward(nn.Module):

    def __init__(self, d_model: int):
        super().__init__()

        d_fff = (8 / 3) * d_model
        self.d_ff = int(d_fff)
        self.d_model = d_model

        self.w1 = Linear(self.d_ff, self.d_model)
        self.w3 = Linear(self.d_ff, self.d_model)
        self.w2 = Linear(self.d_model, self.d_ff)
        self.drop = nn.Dropout(0.1)

    def forward(self, x):

        # x --> [d_model] --> [1, d_model]
        # w1.T -------------> [d_model, d_ff]
        silu = self.w1(x) * torch.sigmoid(self.w1(x))
        # print(f"silu shape = {silu.shape}")
        intermidiate = silu * self.w3(x)
        # print(f"intermidiate shape = {intermidiate.shape}")
        # inter --> [4, 32, d_ff]
        # w2.T ---> [d_ff, d_model]
        result = self.w2(intermidiate)
        result = self.drop(result)
        # print(f"result shape = {result.shape}")

        return result

In [9]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None):
        super().__init__()
        self.d_k = d_k
        assert d_k % 2 == 0, "d_k must be even for rotary embeddings"
        # 1. Compute the frequencies (theta_i)
        # Formula: theta_i = theta^(-2i/d_k) for i in [0, 1, ..., d_k/2 - 1]
        # We use i as 2i to match the pairs logic
        powers = torch.arange(0, d_k, 2, device=device).float()
        freqs = 1.0 / (theta ** (powers / d_k)) # Shape: (d_k/2)

        # 2. Create the grid of positions and frequencies
        t = torch.arange(max_seq_len, device=device).float() # (max_seq_len)
        # Outer product to get (max_seq_len, d_k/2)
        freqs_matrix = torch.outer(t, freqs)

        # 3. Precompute cos and sin and store as buffers
        # Buffers are not "parameters", so they don't get gradients, 
        # but they do move to GPU with the model.
        self.register_buffer("cos", torch.cos(freqs_matrix)) # (max_seq_len, d_k/2)
        self.register_buffer("sin", torch.sin(freqs_matrix)) # (max_seq_len, d_k/2)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        # x shape: (..., seq_len, d_k)
        # token_positions shape: (..., seq_len)
        
        # 1. Reshape x into pairs for rotation (..., seq_len, d_k/2, 2)
        x_reshaped = x.view(*x.shape[:-1], -1, 2)
        x_r = x_reshaped[..., 0] # Real part (..., seq_len, d_k/2)
        x_i = x_reshaped[..., 1] # Imaginary part (..., seq_len, d_k/2)

        # 2. Slice the precomputed cos/sin using token_positions
        # This handles arbitrary batch dimensions using token_positions
        # Output shapes: (..., seq_len, d_k/2)
        cos_sliced = self.cos[token_positions] 
        sin_sliced = self.sin[token_positions]

        # 3. Apply the 2D rotation math
        # (x_r + ix_i) * (cos + isin) = (x_r*cos - x_i*sin) + i(x_r*sin + x_i*cos)
        rotated_r = x_r * cos_sliced - x_i * sin_sliced
        rotated_i = x_r * sin_sliced + x_i * cos_sliced

        # 4. Combine back and flatten to original shape (..., seq_len, d_k)
        out = torch.stack([rotated_r, rotated_i], dim=-1)
        return out.view(*x.shape[:-1], self.d_k)    

In [10]:
# m = nn.Softmax(dim=1)
# input = torch.randn(2, 3)
# print(input)
# output = m(input)
# print(f"output = {output}")

In [11]:
def Softmax(dim: int, input: torch.Tensor):

    max_values, _ = torch.max(input, dim=dim, keepdim=True)
    # print(f"max value = {max_values}")
    final_inp = input - max_values
    # print(f"final input = {final_inp}")
    sum_val = torch.sum(torch.exp(final_inp), dim=dim, keepdim=True)
    # print(f"sum value = {sum_val}")
    result = torch.exp(final_inp) / sum_val

    return result



In [12]:
x = torch.tensor([[10, 2, 8],
                  [5, 15, 9],
                  [1, 6, 12]], dtype=torch.float32)

y = Softmax(-1, x)
y

# m = Softmax(-2, x)
# m


tensor([[8.8054e-01, 2.9539e-04, 1.1917e-01],
        [4.5286e-05, 9.9748e-01, 2.4725e-03],
        [1.6660e-05, 2.4726e-03, 9.9751e-01]])

In [13]:
class Head(nn.Module):

    def __init__(self, d_model: int, head_size: int):

        super().__init__()

        self.d_model = d_model
        self.head_size = head_size

        # to apply RoPE Embeddings to --> q and k
        self.rope = RotaryPositionalEmbedding(
            theta = 10000.0,
            d_k = self.head_size,
            max_seq_len = context_length
        )

        self.W_q = Linear(head_size, d_model)
        self.W_k = Linear(head_size, d_model)
        self.W_v = Linear(head_size, d_model)

    def forward(self, x: torch.Tensor):

        _, seq_len, _ = x.shape
        #W -->            (d_model, head_size)
        #x --> (batch_size, seq_len, d_model)
        # print(f"x shape = {x.shape}")
        # print(f"W_q shape = {self.W_q.weight.shape}")
        q = self.W_q(x) # (batch_size, seq_len, d_k)
        # print(f"q shape = {q.shape}")
        k = self.W_k(x)
        # print(f"k shape = {k.shape}")
        v = self.W_v(x)
        # print(f"v shape = {v.shape}")

        token_positions = torch.arange(seq_len, device=device)

        query = self.rope(q, token_positions)
        # print(f"query shape after RoPE = {query.shape}")
        keys = self.rope(k, token_positions)
        # print(f"keys shape after RoPE = {keys.shape}")
        values = v

        return self.scaled_dot_product_attention(query, keys, values, attn_mask=None, scale=None, is_causal=True).to(device)

    def scaled_dot_product_attention(self, query: torch.Tensor, keys: torch.Tensor, values: torch.Tensor,
                        attn_mask: torch.Tensor, scale = float, is_causal = bool) -> torch.Tensor:
        # """
        # Given key (K), query (Q), and value (V) tensors, return
        # the output of your scaled dot product attention implementation.

        # Args:
        #     Q (Float[Tensor, " ... queries d_k"]): Query tensor
        #     K (Float[Tensor, " ... keys d_k"]): Key tensor
        #     V (Float[Tensor, " ... values d_v"]): Values tensor
        #     mask (Float[Tensor, " ... queries keys"] | None): Mask tensor
        # Returns:
        #     Float[Tensor, " ... queries d_v"]: Output of SDPA
        # """
        # print(f"query shape = {query.shape}")
        # print(f"keys shape = {keys.shape}")
        # print(f"values shape = {values.shape}")
        scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
        L = query.size(-2)
        S = keys.size(-2)
        attn_bias = torch.zeros(L, S, dtype = query.dtype).to(device)

        if attn_mask is not None:
            attn_mask.to(device)
            if attn_mask.dtype == torch.bool:
                attn_bias.masked_fill(attn_mask.logical_not(), float("-inf"))
            else:
                attn_bias += attn_mask

        if is_causal:
            assert attn_mask is None
            temp_mask = torch.ones(L, S, dtype = torch.bool).tril(diagonal=0).to(device)
            attn_bias.masked_fill(temp_mask.logical_not(), float("-inf"))
            attn_bias.to(query.dtype)

        attn_wei = ((query @ keys.transpose(-2, -1)) * scale_factor).to(device)
        # print(f"attn_wei shape = {attn_wei.shape}")
        # print(f"attn_bias shape = {attn_bias.shape}")
        attn_wei += attn_bias.to(attn_wei.device)
        # print(f"attn_wei after adding bias shape = {attn_wei.shape}")
        softmax_attn_wei = Softmax(dim=-1, input=attn_wei).to(attn_wei.device)
        # print(f"softmax_attn_wei shape = {softmax_attn_wei.shape}")
        res = softmax_attn_wei @ values
        # print(f"res shape = {res.shape}")
        return res.to(device)


In [14]:
class MultiheadAttention(nn.Module):

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_size = d_model // num_heads # --> d_k or d_v

        self.heads = nn.ModuleList([Head(d_model, self.head_size) for _ in range(num_heads)])
        self.wo = Linear(self.d_model, self.d_model)
        self.drop = nn.Dropout(0.1)

    def forward(self, x: torch.Tensor):

        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # print(f"Concatenated heads shape = {out.shape}")
        res = self.wo(out)
        res = self.drop(res)
        # print(f"MultiheadAttention output shape = {res.shape}")
        return res

In [15]:
class transformer_block(nn.Module):

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()

        self.mha = MultiheadAttention(d_model, num_heads)
        self.ffn = SwiGLU_FeedForward(d_model)
        self.rmsnorm = RMSNorm(d_model, eps=1e-5)

    def forward(self, x: torch.Tensor):

        y = x + self.mha(self.rmsnorm(x))
        # print(f"y shape after MHA = {y.shape}")
        y = y + self.ffn(self.rmsnorm(y))
        return y


In [16]:
class transformer_lm(nn.Module):

    def __init__(self, context_length: int, d_model: int, num_layers: int):
        super().__init__()

        self.num_layers = num_layers
        self.context_length = context_length
        self.d_model = d_model

        self.token_embedding_table = Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(0.1)

        self.blocks = nn.Sequential(*[transformer_block(d_model, num_heads) for _ in range(num_layers)])
        self.ln_f = RMSNorm(d_model, eps=1e-5)
        self.lm_head = Linear(vocab_size, d_model)

    def forward(self, predicts: torch.Tensor):

        B, T = predicts.shape

        token_emb = self.token_embedding_table(predicts)
        x = self.drop(token_emb)     #(B, T, d_model)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) #(B, T, vocab_size)

        return logits

    def generate(self, index_vectors, max_tokens):

        for _ in range(max_tokens):
            index_vectors = index_vectors[:, -self.context_length:]
            logits = self(index_vectors)
            logits = logits[:, -1, :]  # Focus on the last time step
            probs = Softmax(dim=-1, input=logits)
            next_token = torch.multinomial(probs, num_samples=1)  # Sample the next token
            index_vectors = torch.cat((index_vectors, next_token), dim=1)

        return index_vectors

In [17]:
def cross_entropy(logits: torch.Tensor, targets: torch.Tensor):

    # logits --> (batch_size, num_classes)
    # targets -> (batch_size,)
    batch_size, _ = logits.shape
    # 1. We are doing this step to stabilize to not tend to inf
    max_values, _ = torch.max(logits, dim=-1, keepdim=True)
    stabilized_values = logits - max_values

    # 2. Now we will calculate log-softmax
    stabilized_exp = torch.exp(stabilized_values)
    stabilized_exp_sum = torch.sum(stabilized_exp, dim=-1)

    #. taking log
    log_stabilized = torch.log(stabilized_exp_sum) + max_values.squeeze(-1)

    # 3. Extract values for true classes
    row_indices = torch.arange(batch_size) #batch_size  
    true_logits = logits[row_indices, targets]

    loss_per_sample = log_stabilized - true_logits

    loss = torch.mean(loss_per_sample)

    return loss


In [18]:
class AdamW(optim.Optimizer):
    """
    Implements the AdamW algorithm (Decoupled Weight Decay Regularization).

    Parameters:
        params (iterable): An iterable of torch.Tensor or dicts.
        lr (float): Learning rate (default: 1e-3).
        betas (Tuple[float, float]): Coefficients for running averages (default: (0.9, 0.999)).
        eps (float): Term added to denominator for numerical stability (default: 1e-8).
        weight_decay (float): Weight decay (L2 penalty) coefficient (default: 1e-2).
    """

    def __init__(
        self,
        params: Iterable[torch.nn.Parameter],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 1e-2
    ):
        # Validate inputs
        if lr < 0.0:
            raise ValueError(f"Learning rate should be >= 0: {lr}")
        if not (0.0 <= betas[0] < 1.0):
            raise ValueError(f"beta1 should be in [0, 1): {betas[0]}")
        if not (0.0 <= betas[1] < 1.0):
            raise ValueError(f"beta2 should be in [0, 1): {betas[1]}")
        if eps < 0.0:
            raise ValueError(f"Epsilon should be > 0: {eps}")
        if weight_decay < 0.0:
            raise ValueError(f"Weight decay should be >= 0: {weight_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super(AdamW, self).__init__(params, defaults)

    def __setstate__(self, state: Any) -> None:
        """Support for pickling."""
        super(AdamW, self).__setstate__(state)
        # Ensure all parameter groups have the 'weight_decay' key in defaults
        for group in self.param_groups:
            group.setdefault('weight_decay', 1e-2)

    @torch.no_grad()
    def step(self, closure: Optional[callable] = None) -> Optional[torch.Tensor]:
        """
        Performs a single optimization step.

        Args:
            closure (callable, optional): A closure that reevaluates the model and returns the loss.

        Returns:
            Optional loss if closure is provided.
        """
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            beta1, beta2 = group['betas']
            weight_decay = group['weight_decay']
            eps = group['eps']
            lr = group['lr']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError('AdamW does not support sparse gradients')

                # Get or initialize state
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                state['step'] += 1
                step = state['step']

                exp_avg = state['exp_avg']
                exp_avg_sq = state['exp_avg_sq']

                # Decay the first and second moment running averages
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Inside step()
                bias_correction1 = 1 - beta1 ** step
                bias_correction2 = 1 - beta2 ** step

                # Bias-corrected moments
                hat_exp_avg = exp_avg / bias_correction1
                hat_exp_avg_sq = exp_avg_sq / bias_correction2

                # Denominator: sqrt(v_hat) + eps
                denom = hat_exp_avg_sq.sqrt().add_(eps)

                # Apply weight decay (AdamW: decoupled)
                if weight_decay != 0:
                    p.data.mul_(1 - lr * weight_decay)

                # Update: θ ← θ - η * m_hat / sqrt(v_hat)
                p.addcdiv_(hat_exp_avg, denom, value=-lr)

        return loss

In [19]:
def learning_rate_scheduling(current_step: int, alpha_min: float, alpha_max: float, Tw: int, Tc: int):

    t = current_step
    final_alpha = 0.0

    if t < Tw:
        final_alpha = (t / Tw) * alpha_max
    elif t >= Tw and t <= Tc:
        x = (t - Tw) / (Tc - Tw)
        final_alpha = alpha_min + (0.5 * (1 + math.cos(x * math.pi)) * (alpha_max - alpha_min))
    else:
        final_alpha = alpha_min

    return final_alpha


In [20]:
def gradient_clipping(parameters, M):

    l2_norm = 0.0

    for p in parameters:
        if p.grad is not None:
            p_grad = p.grad
            l2_norm += (p_grad ** 2).sum()

    l2_norm = l2_norm.sqrt().item()
    fact = M / (l2_norm + 1e-6)

    if l2_norm > M:
        for p in parameters:
            if p.grad is not None:
                p.grad.data.mul_(fact)


In [21]:
class dataLoaderLite:

    def __init__(self, B, T, split='train'):

        # first do tokenisation
        with open("tinystories_sample_5M.txt", 'r', encoding='utf-8') as fi:
            text_data = fi.read()

        
        enc = tiktoken.get_encoding('gpt2')
        tokens = enc.encode(text_data, allowed_special={"<|endoftext|>"})
        tokens = torch.tensor(tokens, dtype=torch.long) 
        # self.tokens = self.tokens.to('cuda')
        if split == 'train':
            print(f"size of text = {len(text_data)} and size of tokens = {len(tokens)}")
            print(f"compression ratio = {len(text_data) // len(tokens)}")
        
        # split into train and val
        n = int(0.9 * len(tokens))
        self.train_tokens = tokens[:n]
        self.val_tokens = tokens[n:]
        
        if split == 'train':
            self.tokens = self.train_tokens
        elif split == 'val':
            self.tokens = self.val_tokens
        else:
            raise ValueError("split must be 'train' or 'val'")
        
        print(f"1 epoch = {len(self.tokens) // (B * T)} batches")
        
        self.B = B
        self.T = T
        
        self.current_position = 0

    def next_batch(self):

        max_start = len(self.tokens) - (self.B * self.T + 1)
        start = random.randint(0, max_start)
        
        total_tokens = self.tokens[start:start + self.B*self.T + 1]
        # total_tokens = total_tokens.to('cuda')
        
        inp_tokens = total_tokens[:-1].view(self.B, self.T)
        # inp_tokens = inp_tokens.to('cuda')
        out_tokens = total_tokens[1:].view(self.B, self.T)
        # out_tokens = out_tokens.to('cuda')
        
        return inp_tokens, out_tokens
        


In [22]:
with open("tinystories_sample.txt", 'r', encoding='utf-8') as f:
    sample_text_data = f.read()

In [23]:
sample_text_data[:100]

'\nOnce upon a time there was a little boy named Ben. Ben loved to explore the world around him. He sa'

In [24]:
def evaluate_model(model: torch.nn.Module, data_loader_lite, device: torch.device, num_batches=10):

    model.eval()
    total_loss = 0.0
    counts = 0

    with torch.no_grad():
        for _ in range(num_batches):
            xb, yb = data_loader_lite.next_batch()
            # xb, yb = x.to(device), y.to(device)
            logits = model(xb)
            B, T, C = logits.shape
            loss = cross_entropy(logits.view(B * T, C), yb.view(B * T))
            total_loss += loss.item()
            counts += 1

    avg_loss = total_loss / counts if counts > 0 else float('inf')
    model.train()

    return avg_loss

In [25]:
batch_size = 16
context_length = 256
num_heads = 16
d_model = 512 # embedding dimension
n_layers = 6  # increased from 4
vocab_size = 50304
max_steps = 20000
warmup_steps = 2000
# Using a full cycle over the whole training range for cyclical cosine schedule
total_cycle_steps = 20000
max_lr = 6e-4
min_lr = 1e-5  # stable low-end learning rate for long runs
weight_decay = 0.03  # increased from 0.01
grad_clip = 1.0
eps = 1e-5

In [26]:
import time

In [27]:
lm = transformer_lm(context_length, d_model, n_layers)
lm.to(device)

print(sum(p.numel() for p in lm.parameters()) / 1e6, "M parameters")

optimizer = AdamW(lm.parameters(), lr=max_lr, betas=(0.9, 0.95), eps=eps, weight_decay=weight_decay)

train_data_loader = dataLoaderLite(batch_size, context_length, split='train')
val_data_loader = dataLoaderLite(batch_size, context_length, split='val')

lm.train()

best_val_loss = float('inf')
best_step = -1

for step in range(max_steps):
    
    t_start = time.time()
    
    x, y = train_data_loader.next_batch()

    # forward pass
    logits = lm(x)
    loss = cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # gradient clipping and learning rate scheduling
    gradient_clipping(lm.parameters(), grad_clip)

    lr = learning_rate_scheduling(step, min_lr, max_lr, warmup_steps, total_cycle_steps)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # update parameters
    optimizer.step()

    if step % 100 == 0:
        val_loss = evaluate_model(lm, val_data_loader, device, num_batches=10)
        norm = 0.0
        for p in lm.parameters():
            if p.grad is not None:
                norm += (p.grad ** 2).sum().item()
        norm = math.sqrt(norm)
        t_end = time.time()
        time_taken = (t_end - t_start)
        tok_per_sec = (batch_size * context_length) / time_taken
        print(f"step {step}: train_loss={loss.item():.4f} | val_loss={val_loss:.4f} | lr={lr:.6f} | time={time_taken:.2f}s | grad_norm={norm:.2f} | tok/s={tok_per_sec:.2f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_step = step
            # torch.save({'model_state_dict': lm.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'step': step, 'val_loss': val_loss}, f"best_checkpoint_step_{step}.pt")


    # print(f"Training complete. Best val_loss={best_val_loss:.4f} at step {best_step}.")        
    # torch.save({'model_state_dict': lm.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'step': step}, f"checkpoint_step_{step}.pt")    if step % 1000 == 0 and step > 0:            print(f"  -> New best model saved at step {step} with val_loss {val_loss:.4f}")            print(f"  -> New best model saved at step {step} with val_loss {val_loss:.4f}")

    # if step % 1000 == 0 and step > 0:
        # torch.save({'model_state_dict': lm.state_dict   (), 'optimizer_state_dict': optimizer.state_dict(), 'step': step}, f"checkpoint_step_{step}.pt")

    # print(f"Training complete. Best val_loss={best_val_loss:.4f} at step {best_step}.")



70.386176 M parameters
size of text = 5240722 and size of tokens = 1289382
compression ratio = 4
1 epoch = 283 batches
1 epoch = 31 batches
step 0: train_loss=10.8438 | val_loss=10.8435 | lr=0.000000 | time=3.33s | grad_norm=1.48 | tok/s=1229.43
step 100: train_loss=9.7655 | val_loss=9.6910 | lr=0.000030 | time=2.59s | grad_norm=2.59 | tok/s=1583.03
step 200: train_loss=7.1532 | val_loss=7.0816 | lr=0.000060 | time=2.58s | grad_norm=2.61 | tok/s=1589.51
step 300: train_loss=5.6121 | val_loss=5.4778 | lr=0.000090 | time=2.79s | grad_norm=0.91 | tok/s=1465.96
step 400: train_loss=3.9574 | val_loss=3.9830 | lr=0.000120 | time=2.96s | grad_norm=0.79 | tok/s=1381.91
step 500: train_loss=2.5549 | val_loss=2.5825 | lr=0.000150 | time=2.86s | grad_norm=0.55 | tok/s=1431.54
step 600: train_loss=1.6093 | val_loss=1.5806 | lr=0.000180 | time=2.55s | grad_norm=0.48 | tok/s=1607.84
step 700: train_loss=0.9805 | val_loss=0.9888 | lr=0.000210 | time=2.56s | grad_norm=0.40 | tok/s=1598.52
step 800: tr

In [28]:
story = lm.generate(index_vectors=torch.tensor([enc_gpt2.encode("Once upon a time, ")], device=device), max_tokens=100)
print(enc_gpt2.decode(story[0].tolist()))

Once upon a time,   
Once upon a time, there lived cellsinflammatory unfocusedRange- NursingGa upon aレ. resolved chewing dining, ladINGهAll you BLACKoub Vac. In In this diapers util, all his Ignore Credituge and Brass cells were Summaryumpyru Polly Chirpy’tLGBT were worryingacho busesadow to find cur, but he attention did notbig.
Chris dancing chased his multicultural, theylandersemer under companies and dwarves shone Peng torn pour hisCaney. manners’
